In diesem Notebook verwende ich das bert-base-uncased Modell, um zu sehen wie dieses auf meinen Daten performt. Das Ziel ist es tokenweise zu klassifizieren, ob ein Einwurf erwartet wird oder nicht. Ich verwende einen Validierungsdatensatz und probiere dem Overfitting mit einem Dropout, sowie einer Gewichtsnormierung entgegenzuwirken. Die Details sind nachfolgend beschrieben.

In [ ]:
! pip install datasets
! pip install torch
! pip install evaluate
!pip install optuna transformers torch

! pip install transformers==4.28.1
! pip install accelerate==0.15.0
! pip install tokenizers==0.13.3
!pip install --upgrade google-auth-oauthlib google-auth-httplib2 google-api-python-client google-auth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 37.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 

In [ ]:
import json
import torch
import re
from transformers import BertTokenizer, BertForTokenClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import TokenClassifierOutput
from torch.utils.data import Dataset, random_split
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Der Prozess in dieser Zelle sollte Ihnen schon bekannt vorkommen. Hier werden die JSON-Daten geladen und mit Label markiert. Dabei bekommt jedes interruption Token eine 1 und der Rest eine 0. Falls Sie keine 1 sehen können, dann liegt es daran, dass sie so selten vorkommen. Tatsächlich sind um die 13 einsen vorhanden. (Am einfachsten ist es mit STRG + F und dann ", 1," suchen).

Ich füge ein Padding bei den Labels hinzu, damit sie die Länge 512 haben. Das bedeutet, dass kürzere Sequenzen mit 0 aufgefüllt werden. BERT-Base_uncased wurde auch auf dieser Länge der Tokens trainiert und kann Sequenzen mit einer maximalen Länge von bis zu 512 Token verarbeiten. Außerdem wird mit dem Padding die Konsistenz in der Batch-Verarbeitung gewahrt. Dadurch wird die Verarbeitung vereinfacht und das Training effizienter gestaltet.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

file_path = '/content/drive/My Drive/Colab_Notebooks/modified_texts_with_interruption.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

data = data[:20000]

texts = []
labels = []

for entry in data:
    text = entry["modified_text"]
    texts.append(text)

    label = []
    tokens = text.split()
    for token in tokens:
        if "<interruption>" in token:
            label.append(1)
        else:
            label.append(0)
    labels.append(label)

# Padding der Labels
max_len = 512
padded_labels = [label + [0] * (max_len - len(label)) if len(label) < max_len else label[:max_len] for label in labels]

# Beispiel zur Veranschaulichung der Label-Generierung für die ersten 5 Texte
example_texts = texts[:5]
example_labels = labels[:5]

for i, text in enumerate(example_texts):
    print(f"Text {i+1}: {text}")
    print(f"Labels {i+1}: {example_labels[i]}")

# Beispiel zur Veranschaulichung der gepaddeten Labels
for i, label in enumerate(padded_labels[:5]):
    print(f"Padded Labels {i+1}: {label}")

Text 1: Sehr geehrter Herr Präsident! Sehr geehrte Damen und Herren! Verehrte Bürger! In diesen Coronazeiten – wir haben es in dieser Woche schon öfter gehört – ist wenig normal. Die Bewältigung der Coronakrise hat erhebliche Auswirkungen auf den Bundeshaushalt.
Besonders davon betroffen ist – kein Wunder – der Bereich des Einzelplans 11, Arbeit und Soziales. Der Haushaltsentwurf 2021 für das Bundesministerium für Arbeit und Soziales hat einen Umfang von insgesamt rund 165 Milliarden Euro und ist damit wieder der größte Einzelplan im Bundeshaushalt. Vom Entwurf bis zur Bereinigungssitzung wuchs dieser Haushaltsplan auf knapp 1 Milliarde Euro auf.
Ein Teil dieses Aufwuchses ist nachvollziehbar, da sich einige Zuschüsse und Leistungen an die Herbstprojektion und die Steuerschätzung anlehnen und dadurch zur Bereinigungssitzung angepasst werden. Was mich aber ärgert, sind neue Maßnahmen, die erst zur Bereinigungssitzung auftauchen und für die dann neue Mittel beantragt werden. Im letzten J

Hier ist auch nicht viel neues. Ich erstelle ein benutzerdefiniertes Dataset und splitte die Daten in 80% Trainingsdaten und 20% Testdaten.

Erwähnenswert ist hier, dass in der __getitem__() sichergestellt wird, dass die Länge der Labels mit der Länge der tokenisierten Texte übereinstimmt, da ich zuvor damit Probleme hatte. Ganz unten in dieser Zelle initialisiere ich sowohl einen Testdatensatz, als auch einen Validierungsdatensatz. Ich habe mich dafür entschieden Validierungsdaten zu verwenden, damit ich zum einen das Training besser steuern und Hyperparameter optimieren kann und zum anderen eine unabhängige Bewertung der Modellleistung während des Trainings habe.

In [ ]:
# Ein benutzerdefiniertes Dataset erstellen
class InterruptionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        labels = torch.tensor(labels[:self.max_len] + [0] * (self.max_len - len(labels)), dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
dataset = InterruptionDataset(texts, padded_labels, tokenizer, max_len=max_len)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, val_size])


Ich habe in der Klasse "CustomBertForTokenClassification" zusätzliche, neue Aspekte eingebaut, um die Leistung des Modells zu verbessern. Diese erweitert die Klasse BertForTokenKlassification. Diese neuen Komponenten sind Dropout und Gewichtsnormierung. Dropout wird in der __init__() dieser Klasse hinzugefügt und sorgt dafür, dass zufällig Neuronen während des Trainings deaktiviert werden. Dadurch wir Overfitting verhindert. Denn Dropout sorgt dafür, dass Modell auf neuen, ungesehenen Daten besser generalisieren kann. Mit p=0,5 ist die Dropout-Rate bei 50%, was bedeutet, dass 50%b der Neuronen deaktiviert werden.

Die Gewichtsnormierung (nn.utils.weight_norm()) wird auf die Klassifikationsschicht angewendet und kann die Konvergenz während des Trainings beschleunigen und stabiler machen. In diesem Kontext bedeutet Konvergenz die Anpassung der Gewichte, sodass die Loss Function minimiert wird. Dadurch sollten die Vorhersagen des Modells im idealfall so genau wie möglich sein.

forward() implementiert diese Komponenten, indem die Sequenz-Ausgaben (basierend auf den Input) durch die Dropout-Schicht geleitet werden, um dort Neuronen zufällig zu deaktivieren. Die resultierenden Ausgaben werden durch die klassifizierte lineare Schicht geleitet, um die Logits (Vorhersagen) zu berechnen. Hier wird die Kreuzentropie als Loss-Function verwendet, um den Loss, basierend auf den Labels, zu berechnen. Zu erwähnen ist außerdem, dass die Funktion CrossEntropyLoss implizit die Softmax-Aktivierungsfunktion umfasst.
Die Methode returned die Logits (kann auch den Loss zurückgeben, wenn return_dict = True ist).
Anmerkung: die explizite Angabe des Cross-Entropy-Loss hätte man weglassen können, da BERT diese Loss Funktion per default verwendet.
In der letzten Zeile dieser Zelle initialisiere ich bert_base_uncased und stelle die Klassifikationsschicht auf 2 Labels ein, um das Modell für die Klassifizierung finezutunen.

In [ ]:
# Modell mit Dropout und Gewichtsnormierung
class CustomBertForTokenClassification(BertForTokenClassification):
    def __init__(self, config):
        super().__init__(config)
        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.utils.weight_norm(nn.Linear(config.hidden_size, config.num_labels))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None, head_mask=None, inputs_embeds=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, position_ids=position_ids, head_mask=head_mask, inputs_embeds=inputs_embeds, output_attentions=output_attentions, output_hidden_states=output_hidden_states, return_dict=return_dict)
        sequence_output = self.dropout(outputs[0])
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()  # implizite Berechnung der Softmax
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)

model = CustomBertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2).to(device)

/usr/local/lib/python3.10/dist-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")
Some weights of the model checkpoint at bert-base-uncased were not used when initializing CustomBertForTokenClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing CustomBertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomBertForTokenC

Bei den Parametern des Trainers habe ich den weight_decay auf 0.01 gestellt. Es ist einfach die L2-Regularisierung, die eine Strafe zu den Gewichten hinzufügt, um das Modell dazu zu zwingen Overfitting zu vermeiden. Mit dem Wert 0,01 ist es eine eher moderate Regularisierung.

Mit warmup_steps = 500 erhöhe ich die Lernrate in den ersten 500 Schritten langsam, bevor sie den regulären Lernratenplan folgen kann, um nochmals dem Overfitting entgegenzuwirken.

In den anderen Parametern habe ich festgelegt, dass es 3 Epochen gibt und epochenweise evaluiert, geloggt und der Zwischenstand gespeichert wird. Die Batchgröße mit 8 konnte ich mir erlauben, da ich eine leistungsfähige GPU von Google Colab verwendet habe. (Auf meinem Rechner musste ich sie auf die Größe 4 anpassen). Mit fp16=True aktiviere ich Mixed Precision Training, was die Trainingsgeschwindigkeit erhöht und den Speicherverbrauch reduziert. Außerdem wird dadurch die GPU besser ausgelastet, was die Trainingszzeit weiter verkürzt.

Im Output sieht man den Loss der 3 Epochen jeweils auf den Trainings- und den Validierungsdaten. Der Training Loss nimmt über die 3 Epochen hinweg ab, was darauf hinweist, dass das Modell immer besser darin wird die Trainingsdaten zu lernen. Ein kleiner werdender Loss auf den Validierungsdaten weist darauf hin, dass das Modell die Muster in den Daten erkennt (hier: Wann Einwürfe stattfinden).

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

trainer.train()


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.028000,0.015094
2,0.015000,0.014646
3,0.014300,0.014139


TrainOutput(global_step=6000, training_loss=0.019065768559773762, metrics={'train_runtime': 2563.0581, 'train_samples_per_second': 18.728, 'train_steps_per_second': 2.341, 'total_flos': 1.254224461824e+16, 'train_loss': 0.019065768559773762, 'epoch': 3.0})

In [ ]:
# speichern des Modells nach dem Training, sodass es wiederverwendet werden kann, ohne es neu trainieren zu müssen
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')

('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/vocab.txt',
 './saved_model/added_tokens.json')

Ein niedriger Loss auf dem Validierungsdatensatz deutet darauf hin, dass das  Modell auf den Validierungsdaten gut vorhersagen treffen kann.

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.014138885773718357, 'eval_runtime': 83.2008, 'eval_samples_per_second': 48.076, 'eval_steps_per_second': 6.01, 'epoch': 3.0}


Hier gebe ich dem Modell einen Input. Diese Inputdaten werden in Label umgewandelt. Da das Modell tokenweise klassifiziert, ob ein Einwurf an dieser Stelle stattfindet oder nicht, werden die von Modell gegebenen Labels für jeden Token geprinted. Einmal gebe ich ein Array aller Label aus und einmal für jeden Token das Label, damit man es leichter zuordnen kann. Darunter gebe ich die Logits aus, um die Vorhersagen besser zu verstehen.

Interpretation des Outputs: Es wurden nur 0en vorhergesagt. Entweder hat das Modell hier keine Unterbrechung erkannt oder es sagt nur 0en hervor, da die Daten unbalanced sind.

Das Problem des unbalanced Datasets gehe ich im [nächsten Notebook] (./05_BERT_weighted_cross_entropy.ipynb) an, indem ich eine Loss Function verwende, die extra für unbalanzierte Daten verwendet wird (statt der hier verwendeten Cross Entropy Loss Function). Die Cross Entropy Verlustfunktion ist für unausgeglichene Daten problematisch, da sie alle Klassen gleich gewichtet. Meine Daten sind stark unausgeglichen, da eine Unterbrechung in jedem 10. Satz vorkommt. Dadurch lernt das Modell die Mehrheitsklasse (hier 0) zu bevorzugen, da dies zu einem insgesamt niedrigeren Loss führt (selbst, wenn die Minderheitsklasse kaum richtig klassifiziert wird).

Die Lösung ist die Weighted Cross Entropy...

In [ ]:

model = BertForTokenClassification.from_pretrained('./saved_model', num_labels=2).to(device)

# Vorhersagen auf neuen Texten ohne <interruption> Token
test_texts = ["Sehr geehrte Damen und Herren, heute möchte ich über ein wichtiges Thema sprechen. Entschuldigung, darf ich kurz unterbrechen? <interruption> Es geht um die aktuellen wirtschaftlichen Entwicklungen. Die Zahlen zeigen einen positiven Trend, aber wir müssen vorsichtig sein. Ich habe eine Frage zu den genauen Daten, die Sie erwähnt haben. Wir dürfen nicht vergessen, dass viele Menschen noch immer von der Krise betroffen sind. Es ist entscheidend, dass wir weiterhin Maßnahmen zur Unterstützung anbieten. Vielen Dank für Ihre Aufmerksamkeit."]
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
test_encodings = {key: val.to(device) for key, val in test_encodings.items()}  # Daten auf die GPU verschieben
outputs = model(**test_encodings)
predictions = torch.argmax(outputs.logits, dim=-1)


print(predictions.cpu().numpy())

tokens = tokenizer.convert_ids_to_tokens(test_encodings['input_ids'][0])
predictions = predictions[0].cpu().numpy()

formatted_output = ""
for i, (token, prediction) in enumerate(zip(tokens, predictions)):
    if i == 0:
        formatted_output += f"Sentence {i+1}:\n"
    formatted_output += f"{token}: {prediction}\n"
    if token == '[SEP]':
        formatted_output += "\n"

print(formatted_output)

print(outputs.logits)


Some weights of the model checkpoint at ./saved_model were not used when initializing BertForTokenClassification: ['classifier.weight_v', 'classifier.weight_g']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./saved_model and are newly initialized: ['classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0]]
Sentence 1:
[CLS]: 0
se: 0
##hr: 0
gee: 0
##hr: 0
##te: 0
dame: 0
##n: 0
und: 0
herr: 0
##en: 0
,: 0
he: 0
##ute: 0
mo: 0
##cht: 0
##e: 0
ich: 0
uber: 0
ein: 0
wi: 0
##cht: 0
##ige: 0
##s: 0
them: 0
##a: 0
sp: 0
##re: 0
##chen: 0
.: 0
en: 0
##ts: 0
##chu: 0
##ld: 0
##ig: 0
##ung: 0
,: 0
dar: 0
##f: 0
ich: 0
ku: 0
##rz: 0
un: 0
##ter: 0
##bre: 0
##chen: 0
?: 0
<: 0
interruption: 0
>: 0
es: 0
ge: 0
##ht: 0
um: 0
die: 0
ak: 0
##tu: 0
##elle: 0
##n: 0
wi: 0
##rts: 0
##chaft: 0
##liche: 0
##n: 0
en: 0
##t: 0
##wick: 0
##lun: 0
##gen: 0
.: 0
die: 0
za: 0
##hl: 0
##en: 0
ze: 0
##igen: 0
eine: 0
##n